In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Nạp API key từ file .env
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# Khởi tạo Client với SDK mới
client = genai.Client(api_key=api_key)

try:
    # Sử dụng model gemini-3.8-flash theo khuyến nghị của Google
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents="Nói 'Xin chào!' bằng tiếng anh.",

    )
    print("Kết quả:", response.text)
except Exception as e:
    print("Lỗi khi gọi API:", e)

Kết quả: Bạn có thể nói là: **"Hello!"** hoặc **"Hi!"** nhé.


In [5]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Hãy tóm tắt tác dụng của Docker trong 2 gạch đầu dòng.",
    config=types.GenerateContentConfig(
        temperature=0.2,          # Logic, chính xác, ít lan man 0.0 nghiem tuc => 1.0 bay bong 
        max_output_tokens=100,    # Giới hạn độ dài để tiết kiệm hạn mức token
    )
)

print(response.text)

Dưới đây là tóm tắt tác dụng của Docker trong 2 gạch đầu dòng:

* **Đảm bảo tính nhất quán:** Đóng gói ứng dụng cùng toàn bộ môi trường (thư viện, cấu hình) vào các "container", giúp ứng dụng chạy ổn định trên mọi máy tính mà không gặp lỗi "trên máy tôi thì chạy được".
* **Tối ưu hóa triển khai:** Giúp việc cài đặt, di chuyển và mở rộng


Markdown
# Ghi nhớ: Kỹ thuật Chain-of-Thought (CoT) Prompting

## 1. Bản chất & Cơ chế
* **Định nghĩa:** Bắt LLM trình bày từng bước suy luận trung gian (dàn giáo suy nghĩ) trước khi chốt đáp án cuối cùng, thay vì nhảy cóc ra kết quả ngay.
* **Cơ chế:** LLM sinh chữ bằng cách dự đoán token tiếp theo. Khi viết ra bước 1, chính bước 1 trở thành ngữ cảnh đầu vào giúp mô hình tính toán bước 2 chuẩn xác hơn.
* **Bản chất:** Không nạp thêm kiến thức mới, chỉ "mở khóa" khả năng suy luận logic sẵn có của mô hình.

---

## 2. Hai hình thức triển khai

| Tiêu chí | Zero-shot CoT | Few-shot CoT |
| :--- | :--- | :--- |
| **Bản chất** | Dùng câu "thần chú", không cần ví dụ | Đưa kèm 1–2 bài toán mẫu đã giải sẵn |
| **Cách dùng** | Thêm: *"Hãy suy nghĩ từng bước một rồi đưa ra đáp án."* (*"Let's think step by step."*) | Đưa cặp mẫu `(User: bài toán -> Assistant: các bước giải + đáp án)` trước câu hỏi thật |
| **Ưu điểm** | Nhanh, ngắn gọn, tiết kiệm token | Ép khuôn định dạng và phong cách suy luận chính xác theo ý muốn |
| **Khi nào dùng**| Bài toán logic phổ thông, kiểm tra nhanh | Bài toán đặc thù, cấu trúc phức tạp, cần chuẩn hóa output |

In [6]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Nạp API key từ file .env
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)

# 2. Đề bài toán logic nhiều bước
bai_toan = (
    "Một lớp có 25 học sinh. 3/5 số đó là nữ. "
    "Trong số nữ, 4 bạn đeo kính. "
    "Hỏi có bao nhiêu bạn nữ KHÔNG đeo kính?"
)

# 3. Kỹ thuật Zero-shot CoT: Ghép câu 'thần chú' vào cuối đề bài
prompt_cot = f"{bai_toan}\n\nHãy suy nghĩ từng bước một, rồi đưa ra đáp án cuối cùng."

# 4. Cấu hình system instruction và temperature=0 cho logic ổn định
config = types.GenerateContentConfig(
    system_instruction="Bạn là trợ giảng Toán, luôn trình bày lời giải cẩn thận và rõ ràng từng bước.",
    temperature=0.0,         # Đặt 0 để suy luận logic ổn định, tránh bịa số
    max_output_tokens=600,   # Giới hạn token để tiết kiệm quota
)

try:
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",  # Hoặc "gemini-3.8-flash"
        contents=prompt_cot,
        config=config,
    )
    print("--- LỜI GIẢI CỦA GEMINI ---")
    print(response.text)
except Exception as e:
    print("Lỗi:", e)

--- LỜI GIẢI CỦA GEMINI ---
Chào bạn, mình là trợ giảng Toán của bạn. Chúng ta sẽ cùng giải bài toán này theo từng bước nhé:

**Bước 1: Tính số học sinh nữ trong lớp.**
Theo đề bài, lớp có 25 học sinh và số học sinh nữ chiếm $\frac{3}{5}$ tổng số học sinh.
Số học sinh nữ là:
$25 \times \frac{3}{5} = 15$ (học sinh)

**Bước 2: Tính số học sinh nữ không đeo kính.**
Chúng ta biết rằng trong số 15 bạn nữ này, có 4 bạn đeo kính. Để tìm số bạn nữ không đeo kính, ta lấy tổng số bạn nữ trừ đi số bạn nữ có đeo kính:
$15 - 4 = 11$ (học sinh)

**Đáp số:** Có **11** bạn nữ không đeo kính.


In [7]:
prompt_few_shot = """Bạn là trợ lý giải toán logic, luôn phân tích theo các bước đánh số rõ ràng.

Ví dụ mẫu:
Câu hỏi: Có 12 quả táo, chia đều vào 3 rổ, sau đó bỏ bớt 1 rổ đi. Hỏi còn lại bao nhiêu quả táo?
Lời giải:
Bước 1: 12 chia 3 = 4 quả mỗi rổ.
Bước 2: Bỏ bớt 1 rổ tức là bớt đi 4 quả.
Bước 3: Số táo còn lại là 12 - 4 = 8 quả.
Đáp án: 8 quả.

---
Bây giờ hãy giải bài toán sau:
Câu hỏi: Một cửa hàng giảm giá 20%, sau đó giảm tiếp 10% trên giá đã giảm. Hỏi tổng cộng giảm bao nhiêu phần trăm so với giá gốc?
Lời giải:
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt_few_shot,
    config=types.GenerateContentConfig(temperature=0.0),
)

print(response.text)

Lời giải:

Bước 1: Giả sử giá gốc của sản phẩm là 100%. Sau khi giảm giá 20%, giá của sản phẩm còn lại là: 100% - 20% = 80% (giá gốc).

Bước 2: Tính mức giảm tiếp theo 10% trên giá đã giảm (80%): 10% × 80% = 8%.

Bước 3: Giá cuối cùng của sản phẩm là: 80% - 8% = 72% (giá gốc).

Bước 4: Tổng cộng phần trăm đã giảm so với giá gốc là: 100% - 72% = 28%.

Đáp án: 28%.


# Ghi nhớ: LangChain & LLM Chain Cơ Bản với LCEL

## 1. LangChain là gì?
* **Bản chất:** LangChain không phải mô hình AI thay thế LLM mà là **lớp keo (framework)** giúp ghép nối LLM với Prompt, Output Parser, Tool và Memory thành một dây chuyền làm việc mượt mà.
* **Vấn đề giải quyết:** Thay vì phải nối chuỗi `f"..."` thủ công, tự gọi API và tự bóc tách kết quả từ dict/JSON thô, LangChain chuẩn hóa mọi công đoạn thành các khối lắp ghép.

---

## 2. Ba thành phần lõi (Khuôn mẫu P-L-P)

| Thành phần | Class trong Code | Vai trò |
| :--- | :--- | :--- |
| **Prompt** | `ChatPromptTemplate` | Khuôn mẫu câu hỏi có chừa biến `{bien}` để tái sử dụng linh hoạt nhiều lần |
| **LLM** | `ChatOpenAI` *(hoặc `ChatGoogleGenerativeAI`)* | "Bộ não" nhận prompt và sinh phản hồi |
| **Parser** | `StrOutputParser` | "Người dọn dẹp" bóc tách tin nhắn thô thành chuỗi text thuần túy sạch sẽ |

---

## 3. Cú pháp LCEL (LangChain Expression Language)

* **Toán tử Pipe `|`:** Nối các thành phần theo dòng chảy dữ liệu từ trái sang phải:
  $$\text{Prompt (điền biến)} \longrightarrow \text{LLM (suy nghĩ/trả lời)} \longrightarrow \text{Parser (bóc tách chuỗi)}$$
* **Khuôn mẫu vàng:**
  ```python
  chain = prompt | llm | parser
  ket_qua = chain.invoke({"ten_bien": "gia_tri"})

In [8]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Nạp GOOGLE_API_KEY từ .env
load_dotenv()

# 1. Khuôn mẫu Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Bạn là trợ lý dịch thuật, dịch sang tiếng Anh, chỉ trả về bản dịch."),
    ("human", "Dịch câu này: {cau}")
])

# 2. Khởi tạo LLM Gemini thay cho ChatOpenAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",  # Hoặc "gemini-3.8-flash"
    temperature=0.0
)

# 3. Bộ dọn dẹp output
parser = StrOutputParser()

# 4. Nối dây chuyền bằng LCEL (Pipe |)
chain = prompt | llm | parser

# 5. Chạy chain
ket_qua = chain.invoke({"cau": "Tôi yêu lập trình"})
print(ket_qua)
# Output: I love programming

I love programming.


# Ghi nhớ: Cơ chế Memory Hiện đại (Modern Memory) trong LangChain

## 1. Bản chất cốt lõi: LLM là Stateless
* **Stateless (Không trạng thái):** Mỗi lời gọi API (`invoke()`, `generate_content()`, `create()`) là một HTTP request độc lập. Server mô hình xóa sạch ngữ cảnh ngay sau khi trả lời.
* **Bản chất của Memory:** Mô hình không tự nhớ. "Trí nhớ" thực chất là quy trình lặp ở tầng ứng dụng:
  $$\text{Đọc lịch sử cũ} \longrightarrow \text{Nhét vào prompt} \longrightarrow \text{LLM trả lời} \longrightarrow \text{Ghi (input, output) mới vào sổ}$$

---

## 2. Classic Memory (Deprecated) vs Modern Approach

| Tiêu chí | Classic Memory (Đã khai tử) | Modern Memory (Hiện đại đang dùng) |
| :--- | :--- | :--- |
| **Các class đại diện** | `ConversationBufferMemory`, `ConversationSummaryMemory`, `ConversationChain` | `MessagesPlaceholder`, `RunnableWithMessageHistory`, `LangGraph State` |
| **Trạng thái hỗ trợ** | Bị **deprecated** từ LangChain 0.3.x | Chuẩn khuyến nghị chính thức (Standard) |
| **Kiến trúc luồng** | Ôm đồm, buộc dùng với `ConversationChain` cũ | Tách bạch giữa **Lưu trữ (Store)** và **Dây chuyền LCEL (`prompt \| llm`)** |
| **Hỗ trợ đa người dùng** | Khó cô lập nhiều phiên chat cùng lúc | Tự nhiên và chuẩn hóa qua tham số `session_id` |
| **Khả năng mở rộng** | Chủ yếu giữ trong RAM | Dễ dàng gắn cắm (pluggable) với Redis, MongoDB, Postgres |

---

## 3. Ba thành phần cấu tạo Modern Memory

1. **`MessagesPlaceholder(variable_name="history")`:** Một vị trí trống cắm sẵn bên trong `ChatPromptTemplate` để động nạp danh sách tin nhắn cũ vào prompt.
2. **Hàm `get_session_history(session_id)`:** Quản lý kho lưu trữ (`store`), trả về đối tượng `ChatMessageHistory` tương ứng với từng người dùng cụ thể.
3. **`RunnableWithMessageHistory`:** Wrapper bọc ngoài LCEL chain, tự động đọc lịch sử từ kho đưa vào placeholder và tự động ghi câu trả lời mới vào kho sau khi chạy xong.

---

In [9]:
import os
from dotenv import load_dotenv
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser  # 1. Import Parser
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.0
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Bạn là trợ lý thân thiện, trả lời ngắn gọn."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# 2. Gắn parser vào chuỗi LCEL: prompt | llm | parser
chain = prompt | llm | StrOutputParser()

store = {}

def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 3. Bọc chain có parser
chat = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history"
)

cfg = {"configurable": {"session_id": "quyen-01"}}

# Lượt 1: Kết quả trả về trực tiếp là string, không cần .content
res1 = chat.invoke({"input": "Tớ tên Quyên"}, config=cfg)
print("Lượt 1:", res1)

# Lượt 2:
res2 = chat.invoke({"input": "Tớ tên gì?"}, config=cfg)
print("Lượt 2:", res2)

C:\Users\kedie\AppData\Local\Temp\ipykernel_24164\2248166579.py:29: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


Lượt 1: Chào Quyên! Rất vui được làm quen với bạn. Mình có thể giúp gì cho bạn hôm nay không?
Lượt 2: Bạn tên là Quyên!
